# `confit` package tour

Run the cells top-to-bottom. Cells marked **[GPU]** need a GPU and will try to download ESM weights — skip them if you just want to read logic. Everything else runs on CPU instantly.

| Section | Module |
|---------|--------|
| 1 | `config/` — YAML → validated config |
| 2 | `models/registry.py` — model name → HuggingFace ID |
| 3 | `models/scaling.py` — AModule strategies (the `a` coefficient) |
| 4 | `losses/` — BradleyTerry + KL regularisation |
| 5 | `scoring/masked_marginal.py` — the core forward pass (mock model) |
| 6 | `data/dataset.py` — loading real mutation data |
| 7 | `training/evaluator.py` — EvaluationResult |
| 8 | `training/trainer.py` — TrainMode, ConFitTrainer structure |
| 9 | `runners/experiment.py` — combo grid & RunMode |
| 10 | `runners/inference.py` — InferenceAggregator |
| 11 | `baselines/spurs_ridge.py` — Ridge baseline |
| 12 | `metrics/` + `utils/` — Spearman, seeding, cleanup |

In [1]:
import sys, os
sys.path.insert(0, "/work/shannon/fine-tune_plm")
os.chdir("/work/shannon/fine-tune_plm")

import torch
import numpy as np
import pandas as pd
from pathlib import Path
print("Setup done. torch:", torch.__version__)

Setup done. torch: 2.9.1+cu128


---
## 1 · `config/` — YAML → validated TrainingConfig

**Files:** `confit/config/schema.py`, `confit/config/loader.py`

`ConfigLoader.load()` reads the YAML file and returns a frozen Pydantic dataclass.
All fields are type-validated at load time — you'll get a clear error if the YAML has wrong types or violates constraints (e.g. `min_lr > ini_lr`).

In [3]:
from confit.config.loader import ConfigLoader
from confit.config.schema import TrainingConfig

cfg = ConfigLoader.load("config/training_config.yaml")
print(cfg)
print()
print("model          :", cfg.model)
print("shot           :", cfg.shot)
print("lora_r         :", cfg.lora_r)
print("lora_alpha     :", cfg.lora_alpha)
print("max_epochs     :", cfg.max_epochs)
print("lambda_reg     :", cfg.lambda_reg)
print("per_device_bs  :", cfg.per_device_batch_size)  # derived property: batch_size // gpu_number

model='ESM-1v' batch_size=16 gpu_number=3 lora_r=8 lora_alpha=8 lora_dropout=0.1 ini_lr=0.0005 min_lr=0.0001 max_epochs=30 lambda_reg=0.1 shot=64 endure_time=5

model          : ESM-1v
shot           : 64
lora_r         : 8
lora_alpha     : 8
max_epochs     : 30
lambda_reg     : 0.1
per_device_bs  : 5


In [ ]:
# Pydantic catches bad values immediately at construction time.
# Uncomment to see the ValidationError it raises:

# bad_cfg = TrainingConfig(
#     model="ESM-1v", batch_size=16, gpu_number=3,
#     lora_r=8, lora_alpha=8, lora_dropout=0.1,
#     ini_lr=1e-4, min_lr=5e-3,   # ← min_lr > ini_lr → error
#     max_epochs=30, lambda_reg=0.1, shot=64, endure_time=5,
# )

---
## 2 · `models/registry.py` — model name → HuggingFace ID

**Files:** `confit/models/registry.py`, `confit/models/factory.py`

`ModelRegistry` is a pure lookup table. No model is downloaded here — it just maps the string you put in the YAML to a HuggingFace hub ID.

ESM-1v is special: it has 5 pre-trained ensemble seeds (1–5). The seed is appended to the hub ID.

In [4]:
from confit.models.registry import ModelRegistry, ModelVariant

# Show all supported variants
for v in ModelVariant:
    print(f"{v.value:<10} → {ModelRegistry.hub_id(v, model_seed=1)}")

print()
# ESM-1v has 5 seeds
for seed in range(1, 6):
    print(f"ESM-1v seed {seed} → {ModelRegistry.hub_id(ModelVariant.ESM_1V, seed)}")

print()
# Resolve from a config string
variant = ModelRegistry.from_string(cfg.model)
print(f"cfg.model='{cfg.model}' resolves to: {variant}")

ESM-1v     → facebook/esm1v_t33_650M_UR90S_1
ESM-1b     → facebook/esm1b_t33_650M_UR50S
ESM-2      → facebook/esm2_t48_15B_UR50D

ESM-1v seed 1 → facebook/esm1v_t33_650M_UR90S_1
ESM-1v seed 2 → facebook/esm1v_t33_650M_UR90S_2
ESM-1v seed 3 → facebook/esm1v_t33_650M_UR90S_3
ESM-1v seed 4 → facebook/esm1v_t33_650M_UR90S_4
ESM-1v seed 5 → facebook/esm1v_t33_650M_UR90S_5

cfg.model='ESM-1v' resolves to: ModelVariant.ESM_1V


In [ ]:
# ESMModelFactory builds:
#   backbone  — the trainable model (LoRA will be applied to this)
#   reg_model — a frozen copy of the SAME checkpoint (used for KL regularisation)
#   tokenizer — matching ESM tokenizer
# 
# [GPU] Uncomment to actually download + load (~1.3 GB for ESM-1v):

# from confit.models.factory import ESMModelFactory
# factory = ESMModelFactory()
# bundle = factory.build(ModelVariant.ESM_1V, model_seed=1)
# print(bundle.backbone)
# print("reg_model grad:", any(p.requires_grad for p in bundle.reg_model.parameters()))  # → False

---
## 3 · `models/scaling.py` — AModule strategies

**File:** `confit/models/scaling.py`

This is the learnable **SPURS-DDG correction module**. The idea:
- ESM gives you a masked-marginal score for each mutation.
- SPURS gives you a DDG (free energy) prediction per mutation.
- `AModule` learns *how much* to trust the DDG correction via a scalar `a`.

Four strategies control how `a` is computed:

| `a_type` | Parameters | What it does |
|---|---|---|
| `none` | 0 | No correction, pure ESM |
| `single` | 1 scalar | Same `a` for every position |
| `position-specific` | L scalars | One `a` per position in the protein |
| `context-specific` | MLP | MLP predicts `a` from (ESM logits, DDG features) |

In [5]:
from confit.models.scaling import AModule, ScalingMode

# Toy DDG shape: protein of length 50, 20 amino acids
L, AA = 50, 20
spurs_shape = (L, AA)

for mode in ["none", "single", "position-specific", "context-specific"]:
    A = AModule(mode=mode, spurs_ddg_shape=spurs_shape, a_init=0.1)
    n_params = sum(p.numel() for p in A.parameters())
    print(f"  mode={mode:<20}  ScalingMode={A.mode.value:<20}  trainable_params={n_params}")

  mode=none                  ScalingMode=none                  trainable_params=0
  mode=single                ScalingMode=single                trainable_params=1
  mode=position-specific     ScalingMode=position-specific     trainable_params=50
  mode=context-specific      ScalingMode=context-specific      trainable_params=861


In [6]:
# Show what each strategy's forward() returns
B = 4  # batch size
esm_logits  = torch.randn(B, AA)  # ESM logit slice at mutation positions
ddg_features = torch.randn(B, AA)  # SPURS DDG slice at mutation positions
mut_pos     = torch.tensor([3, 10, 25, 40])  # mutation positions (0-indexed)

print("none              →", AModule("none", spurs_shape, 0.1)(mut_pos=mut_pos))

a_single = AModule("single", spurs_shape, 0.1)
print("single            →", a_single())  # just returns the scalar parameter

a_pos = AModule("position-specific", spurs_shape, 0.1)
print("position-specific →", a_pos(mut_pos=mut_pos))  # looks up A[mut_pos]

a_ctx = AModule("context-specific", spurs_shape, 0.1)
result = a_ctx(esm_i=esm_logits, ddg_i=ddg_features)
print("context-specific  → shape", result.shape)  # MLP output: (B, 1)

none              → None
single            → Parameter containing:
tensor(0.1000, requires_grad=True)
position-specific → tensor([0.1000, 0.1000, 0.1000, 0.1000], grad_fn=<IndexBackward0>)
context-specific  → shape torch.Size([4, 1])


In [ ]:
# combined_way controls WHERE the DDG correction is applied:
#   'scores'  → correction added AFTER masked-marginal score is computed
#               final_score = esm_score + a * ddg_value
#   'logits'  → correction added BEFORE softmax, inside the logit tensor
#               corrected_logits = esm_logits + a * spurs_ddg
#               then compute log-probs from corrected_logits

A_scores = AModule("single", spurs_shape, a_init=0.1, combined_way="scores")
A_logits = AModule("single", spurs_shape, a_init=0.1, combined_way="logits")
print("combined_way='scores':", A_scores.combined_way)
print("combined_way='logits':", A_logits.combined_way)

---
## 4 · `losses/` — BradleyTerry + KL regularisation

**Files:** `confit/losses/bradley_terry.py`, `confit/losses/kl_regularization.py`

**Training loss = BradleyTerry(predicted, golden) + λ * KL(fine-tuned, frozen)**

- **BradleyTerry**: pairwise ranking loss. For every pair (i, j) where `golden[i] > golden[j]`, it penalises if `predicted[i] < predicted[j]`. You don't need absolute fitness values — just relative ordering.
- **KL**: keeps the fine-tuned ESM from drifting too far from the original frozen checkpoint. Prevents catastrophic forgetting.

In [ ]:
from confit.losses.bradley_terry import BradleyTerryLoss

bt_loss = BradleyTerryLoss()

# Perfect ranking: predicted scores match golden ordering
golden    = torch.tensor([3.0, 1.0, 2.0])
predicted_good = torch.tensor([3.0, 1.0, 2.0])  # same order → low loss
predicted_bad  = torch.tensor([1.0, 3.0, 2.0])  # wrong order → high loss

print("Loss (perfect ranking) :", bt_loss(predicted_good, golden).item())
print("Loss (wrong ranking)   :", bt_loss(predicted_bad,  golden).item())

# Edge case: constant prediction (no ordering information)
predicted_flat = torch.tensor([0.0, 0.0, 0.0])
print("Loss (all same)        :", bt_loss(predicted_flat, golden).item())

In [ ]:
from confit.losses.kl_regularization import KLRegularizationLoss

kl_loss = KLRegularizationLoss()

# Toy: batch of 2 sequences, length 10, vocabulary size 33
B, L, V = 2, 10, 33
logits_finetuned = torch.randn(B, L, V)
logits_frozen    = torch.randn(B, L, V)
seq_tokens  = torch.randint(0, V, (B, L))
att_mask    = torch.ones(B, L)  # all tokens valid

loss_diff = kl_loss(logits_finetuned, logits_frozen, seq_tokens, att_mask)
print("KL loss (different models):", loss_diff.item())

# When fine-tuned == frozen, KL should be near 0
loss_same = kl_loss(logits_frozen, logits_frozen, seq_tokens, att_mask)
print("KL loss (same model)      :", loss_same.item())  # ≈ 0

# lambda_reg controls the trade-off
lambda_reg = cfg.lambda_reg
total_loss = bt_loss(predicted_good, golden) + lambda_reg * loss_diff
print(f"\nFull training loss = BT + {lambda_reg} * KL = {total_loss.item():.4f}")

---
## 5 · `scoring/masked_marginal.py` — the core forward pass

**File:** `confit/scoring/masked_marginal.py`

This is the heart of ConFit. For each mutant in a batch:

1. Clone the token sequence and mask the mutation position with `[MASK]`.
2. Run ESM to get per-position logits.
3. (If `combined_way='logits'`): add `a * SPURS_DDG` to the logit tensor.
4. Compute `log P(mutant aa) − log P(wild-type aa)` at the masked position.
5. (If `combined_way='scores'`): add `a * DDG_value` to the score.

Here we use a **mock model** (random logits) to illustrate the logic without needing a GPU.

In [ ]:
from confit.scoring.masked_marginal import MaskedMarginalScorer
from confit.models.scaling import AModule

# Toy protein: length 10, vocab size 33 (ESM uses 33-token vocab)
SEQ_LEN = 10
VOCAB   = 33
BATCH   = 3
MASK_ID = 32  # ESM mask token id

# --- Mock ESM model: just returns random logits -------------------------
class MockESM:
    """Stand-in for a real ESM model — returns random logits."""
    def __call__(self, input_ids, attention_mask, output_hidden_states=False):
        class Out:
            logits = torch.randn(BATCH, SEQ_LEN + 2, VOCAB)  # +2 for BOS/EOS
        return Out()

mock_model = MockESM()

# --- Inputs -------------------------------------------------------------
# 20 canonical amino acid token IDs (positions 4–23 in ESM vocab)
AA_TOKEN_IDS = torch.arange(4, 24)  # shape (20,)

# Random mutant token sequences (BOS + sequence + EOS)
seq  = torch.randint(4, 24, (BATCH, SEQ_LEN + 2))
wt   = torch.randint(4, 24, (BATCH, SEQ_LEN + 2))  # wild-type tokens
mask = torch.ones(BATCH, SEQ_LEN + 2, dtype=torch.long)
# Mutation positions (0-indexed relative to raw sequence, without BOS)
pos  = [torch.tensor([3]), torch.tensor([7]), torch.tensor([1])]

# SPURS DDG tensor: (L, 20)
spurs_ddg = torch.randn(SEQ_LEN, 20)

# --- Build scorer -------------------------------------------------------
class FakeTokenizer:
    mask_token_id = MASK_ID

A = AModule("single", spurs_ddg.shape, a_init=0.1, combined_way="scores")
scorer = MaskedMarginalScorer(
    tokenizer=FakeTokenizer(),
    a_module=A,
    spurs_ddg=spurs_ddg,
    aa_token_ids=AA_TOKEN_IDS,
)

scores, logits = scorer.score(mock_model, seq, mask, wt, pos)
print("scores shape:", scores.shape)  # (B,)
print("logits shape:", logits.shape)  # (B, L+2, V)
print("scores      :", scores)

In [ ]:
# Verify: mutation position IS masked in the input to ESM
# The scorer internally replaces pos+1 (offset by BOS token) with MASK_ID
# We can see this by patching the mock model to record its input:

received_input = {}

class InspectESM:
    def __call__(self, input_ids, attention_mask, output_hidden_states=False):
        received_input["ids"] = input_ids.clone()
        class Out:
            logits = torch.randn(BATCH, SEQ_LEN + 2, VOCAB)
        return Out()

scorer2 = MaskedMarginalScorer(
    tokenizer=FakeTokenizer(), a_module=A,
    spurs_ddg=spurs_ddg, aa_token_ids=AA_TOKEN_IDS,
)
scorer2.score(InspectESM(), seq, mask, wt, pos)

for i, p in enumerate(pos):
    col = p[0].item() + 1  # +1 for BOS
    original = seq[i, col].item()
    after    = received_input["ids"][i, col].item()
    print(f"  sample {i}: pos={p[0].item()}, column={col}, original_token={original}, after_masking={after} (MASK_ID={MASK_ID})")

---
## 6 · `data/dataset.py` — loading real mutation data

**File:** `confit/data/dataset.py`

`MutationDataset` wraps a CSV of mutations into a PyTorch Dataset. Each row is:

| Tensor | What it is |
|---|---|
| `seq` | Tokenised mutant sequence (incl. BOS/EOS) |
| `att_mask` | Attention mask (all 1s) |
| `wt` | Tokenised wild-type sequence |
| `wt_mask` | Wild-type attention mask |
| `pos` | List of mutation positions (0-indexed) |
| `score` | `log_fitness` label |
| `pid` | Numeric sequence ID |
| `mut_id` | Row index in the dataset |

**[GPU]** This cell requires the ESM tokenizer. It downloads ~1 MB of vocab files.

In [ ]:
# [GPU] Uncomment to run — needs ESM tokenizer download (~1 MB)

# from transformers import EsmTokenizer
# from confit.data.dataset import MutationDataset
# from torch.utils.data import DataLoader
#
# DATASET = "A0A140D2T1_ZIKV_Sourisseau_2019"
# tokenizer = EsmTokenizer.from_pretrained("facebook/esm1v_t33_650M_UR90S_1")
#
# train_df = pd.read_csv(f"data/{DATASET}/train_1.csv")
# test_df  = pd.read_csv(f"data/{DATASET}/test.csv")
#
# trainset = MutationDataset(data=train_df, fname=DATASET, tokenizer=tokenizer)
# testset  = MutationDataset(data=test_df,  fname=DATASET, tokenizer=tokenizer)
#
# print(f"Train size: {len(trainset)}, Test size: {len(testset)}")
#
# sample = trainset[0]
# seq_ids, att_mask, wt_ids, wt_mask, pos, score, pid, mut_id = sample
# print("seq token IDs :", seq_ids[:10], "...")
# print("mutation pos  :", pos)
# print("log_fitness   :", score.item())
#
# loader = DataLoader(trainset, batch_size=4, collate_fn=trainset.collate_fn)
# batch  = next(iter(loader))
# seq_b, mask_b, wt_b, wt_mask_b, pos_b, score_b, pid_b, mut_b = batch
# print("batch seq shape:", seq_b.shape)  # (4, L+2)

In [ ]:
# No tokenizer needed — just read the raw CSV to understand the schema
DATASET = "A0A140D2T1_ZIKV_Sourisseau_2019"

train_df = pd.read_csv(f"data/{DATASET}/train_1.csv")
test_df  = pd.read_csv(f"data/{DATASET}/test.csv")

print("train_1.csv columns:", list(train_df.columns))
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")
print()
display(train_df[["mutant", "mutated_position", "log_fitness", "n_mut"]].head(5))

# mutated_position is 0-indexed; seq column has the FULL mutant amino acid sequence
print("\nsequence length:", len(train_df.loc[0, "seq"]))

# Log-fitness distribution
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
train_df["log_fitness"].hist(bins=40, ax=axes[0])
axes[0].set_title("log_fitness distribution (train)")
train_df["n_mut"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("# mutations per variant")
plt.tight_layout()

In [ ]:
# The 5-fold split: train_1 through train_5.
# During training, model_seed=K means fold K is the validation set,
# and the other 4 folds are combined as the training set.

fold_sizes = {}
for k in range(1, 6):
    fold_sizes[f"fold {k}"] = len(pd.read_csv(f"data/{DATASET}/train_{k}.csv"))

fold_sizes["test"] = len(test_df)
pd.Series(fold_sizes).rename("rows").to_frame()

In [ ]:
# SPURS DDG file: rows = protein positions, cols = 20 amino acids
spurs = pd.read_csv(f"data/{DATASET}/spurs_prediction.tsv", sep="\t", index_col=0)
print("spurs_prediction.tsv shape:", spurs.shape)  # (seq_len, 20)
print("columns (amino acids):", list(spurs.columns[:5]), "...")
display(spurs.iloc[:3, :5])  # DDG values: negative = destabilising

---
## 7 · `training/evaluator.py` — EvaluationResult

**File:** `confit/training/evaluator.py`

`ConFitEvaluator` runs a full pass over a DataLoader and returns an `EvaluationResult` dataclass.

The evaluator is decoupled from the trainer — you can call it standalone for test-set evaluation or hyperparameter search.

In [ ]:
from confit.training.evaluator import EvaluationResult
from confit.metrics.correlation import spearman

# Simulate what the evaluator collects after a full eval loop
np.random.seed(42)
n = 100
ground_truth = np.random.randn(n)
# Good model: predictions correlate with ground truth
predicted = ground_truth + 0.3 * np.random.randn(n)

sr = spearman(predicted, ground_truth)

result = EvaluationResult(
    spearman_correlation=sr,
    scores=predicted,
    ground_truth=ground_truth,
    mutation_ids=np.arange(n),
)

print(f"Spearman ρ: {result.spearman_correlation:.4f}")
print(f"scores shape  : {result.scores.shape}")
print(f"gt shape      : {result.ground_truth.shape}")

# is_test=True also collects sequence_ids (PIDs) for writing pred.csv
result_test = EvaluationResult(
    spearman_correlation=sr,
    scores=predicted,
    ground_truth=ground_truth,
    mutation_ids=np.arange(n),
    sequence_ids=np.arange(n),  # populated only in test mode
)
print(f"sequence_ids : {result_test.sequence_ids[:5]} ...")

In [ ]:
# [GPU] Uncomment to run the real evaluator against a trained model:
#
# from confit.training.evaluator import ConFitEvaluator
# from confit.scoring.masked_marginal import MaskedMarginalScorer
#
# scorer    = MaskedMarginalScorer(tokenizer, A, spurs_ddg, aa_token_ids)
# evaluator = ConFitEvaluator(scorer=scorer, accelerator=accelerator, tokenizer=tokenizer)
#
# result = evaluator.evaluate(model, val_loader, is_test=False)
# print(result.spearman_correlation)
#
# result_test = evaluator.evaluate(model, test_loader, is_test=True)
# # result_test.scores, .ground_truth, .mutation_ids, .sequence_ids are all populated

---
## 8 · `training/trainer.py` — ConFitTrainer

**File:** `confit/training/trainer.py`

Two training modes:

| `train_mode` | What happens |
|---|---|
| `full` | Jointly trains LoRA weights + A module from scratch |
| `a_only` | **Stage 1**: trains LoRA only (A frozen). **Stage 2**: freezes LoRA, trains A only. |

The trainer uses **CosineAnnealingWarmRestarts** as the LR scheduler and **early stopping** with patience = `endure_time` epochs.

In [ ]:
from confit.training.trainer import ConFitTrainer, TrainMode

print("Available TrainModes:")
for m in TrainMode:
    print(f"  {m.value}")

print()
print("LoRA target modules :", ConFitTrainer._LORA_TARGET_MODULES)  # ['query', 'value']
print("Stage 2 max epochs  :", ConFitTrainer._STAGE2_EPOCHS)
print("Stage 2 T0 (cosine) :", ConFitTrainer._STAGE2_T0)

print()
# Build optimizer/scheduler to see the structure (no model needed)
dummy_params = [torch.nn.Parameter(torch.randn(10))]
opt = torch.optim.AdamW(dummy_params, lr=cfg.ini_lr)
sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt, T_0=2 * cfg.max_epochs, eta_min=cfg.min_lr
)

lrs = []
for _ in range(cfg.max_epochs * 2):
    lrs.append(sched.get_last_lr()[0])
    sched.step()

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3))
plt.plot(lrs)
plt.axhline(cfg.ini_lr, linestyle="--", color="red",  label=f"ini_lr={cfg.ini_lr}")
plt.axhline(cfg.min_lr, linestyle="--", color="green", label=f"min_lr={cfg.min_lr}")
plt.title(f"LR schedule: CosineAnnealingWarmRestarts  T_0={2*cfg.max_epochs}")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.legend()
plt.tight_layout()

In [ ]:
# [GPU] Full training call (requires ESM model + accelerate setup):
#
# trainer = ConFitTrainer(
#     config=cfg, model=model, model_reg=model_reg, a_module=A,
#     scorer=scorer, evaluator=evaluator, accelerator=accelerator,
#     train_mode=TrainMode.FULL, tokenizer=tokenizer, basemodel=bundle.backbone,
# )
# best_val_sr = trainer.fit(train_loader, val_loader, save_dir=Path("checkpoint_test/..."))
# print("Best validation Spearman:", best_val_sr)

---
## 9 · `runners/experiment.py` — combo grid & RunMode

**File:** `confit/runners/experiment.py`

The experiment runner is the outer orchestration loop. It:
1. Reads ProteinGym dataset metadata (from fasta files).
2. Builds the hyperparameter grid (all `a_type × a_init × combined_way × train_mode` combos).
3. Skips already-completed runs (checks for `pred.csv` + non-empty checkpoint).
4. Calls `accelerate launch confit/train_entry.py ...` as a subprocess for each combo.

Here we inspect the combo grid without touching the filesystem.

In [ ]:
from confit.runners.experiment import ExperimentRunner, RunnerConfig, RunMode

# Show all RunModes
print("RunModes:")
for m in RunMode:
    print(f"  {m.value}")

# Inspect the full combo grid (no filesystem needed)
cfg_run = RunnerConfig(quarter=1, mode=RunMode.MAIN, shot=96)
runner  = ExperimentRunner(cfg_run)

combos, shot, skip_done = runner._build_combos()

print(f"\nTotal combos (MAIN mode): {len(combos)}")
print(f"shot={shot}, skip_done={skip_done}")

combo_df = pd.DataFrame(combos, columns=["a_type","a_init","combined_way","train_mode"])
display(combo_df)

In [ ]:
# None mode is special: only one combo survives (combined_way='scores', train_mode='full', a_init=0.1)
# This is because with no scaling, combined_way and train_mode don't change the result
print("Combos where a_type='none':")
display(combo_df[combo_df.a_type == "none"])

print("\nCombos per a_type:")
display(combo_df["a_type"].value_counts().to_frame())

print("\nCombos per train_mode:")
display(combo_df["train_mode"].value_counts().to_frame())

In [ ]:
# Show what combos each special RunMode selects
for mode in [RunMode.RERUN_SHOT64, RunMode.RERUN_SHOT96, RunMode.RERUN_ALLNONE_FAILED]:
    r = RunnerConfig(quarter=1, mode=mode, shot=96)
    combos_m, shot_m, _ = ExperimentRunner(r)._build_combos()
    print(f"  {mode.value:<25} → {len(combos_m)} combos, shot={shot_m}")

In [ ]:
# Show the expected output paths for a given dataset + combo
runner2 = ExperimentRunner(RunnerConfig(quarter=1, run_suffix="rerun_fixed"))

combo_ex = ("position-specific", -1.0, "logits", "full")
pred_path = runner2._predicted_folder("PTEN_HUMAN", shot=96, seed=1, *combo_ex)
ckpt_path = runner2._checkpoint_folder("PTEN_HUMAN", shot=96, seed=1, *combo_ex)

print("predicted folder:", pred_path)
print("checkpoint folder:", ckpt_path)
print("pred.csv exists:", (pred_path / "pred.csv").exists())
print("checkpoint exists:", ckpt_path.exists())

---
## 10 · `runners/inference.py` — InferenceAggregator

**File:** `confit/runners/inference.py`

After training, each `(dataset, seed)` pair produces a `pred.csv`. The aggregator:
1. Reads all per-seed prediction files.
2. Averages across seeds → ensemble score.
3. Optionally blends with VAE-ELBO retrieval: `α * ensemble + (1-α) * elbo`.
4. Computes Spearman ρ vs ground truth.
5. Writes `results/<dataset>/summary.csv`.

In [ ]:
from confit.runners.inference import InferenceAggregator

agg = InferenceAggregator(alpha=0.8)
print("predicted_dir:", agg.predicted_dir)
print("data_dir     :", agg.data_dir)
print("results_dir  :", agg.results_dir)
print("alpha        :", agg.alpha)

# Show retrieval blending formula:
alpha = 0.8
ensemble_avg = np.array([0.5, 0.3, 0.8])
elbo         = np.array([0.4, 0.6, 0.7])
retrieval    = alpha * ensemble_avg + (1 - alpha) * elbo
print(f"\nBlend formula: {alpha} * ensemble + {1-alpha} * elbo")
for i in range(3):
    print(f"  sample {i}: {alpha}*{ensemble_avg[i]:.1f} + {1-alpha}*{elbo[i]:.1f} = {retrieval[i]:.3f}")

In [ ]:
# Simulate what aggregate() does internally with a mock pred.csv
from confit.metrics.correlation import spearman

np.random.seed(0)
n = 50
truth = np.random.randn(n)

# 5 seeds, each with slightly different predictions
mock_pred = pd.DataFrame({
    "1": truth + 0.4 * np.random.randn(n),
    "2": truth + 0.4 * np.random.randn(n),
    "3": truth + 0.4 * np.random.randn(n),
    "4": truth + 0.4 * np.random.randn(n),
    "5": truth + 0.4 * np.random.randn(n),
    "PID": np.arange(n),
})

seed_cols   = [c for c in mock_pred.columns if c in ["1","2","3","4","5"]]
ensemble    = mock_pred[seed_cols].mean(axis=1).values

per_seed_sr = {c: spearman(mock_pred[c].values, truth) for c in seed_cols}
ensemble_sr = spearman(ensemble, truth)

print("Per-seed Spearman ρ:")
for s, r in per_seed_sr.items():
    print(f"  seed {s}: {r:.4f}")
print(f"\nEnsemble average Spearman ρ: {ensemble_sr:.4f}")
print("(Ensemble almost always beats any single seed — averaging reduces noise)")

---
## 11 · `baselines/spurs_ridge.py` — SPURS Ridge baseline

**File:** `confit/baselines/spurs_ridge.py`

The Ridge baseline competes with ConFit. It uses:
- **One-hot encoding** of the full mutant sequence (`L × 20` features)
- **DeepSequence log pseudo-likelihood** (1 feature)
- **SPURS DDG** (1 feature)

Ridge is trained at multiple N values (48, 96, 144, 192, 240) with 20 random repeats each.

In [ ]:
from confit.baselines.spurs_ridge import _FeatureBuilder, MutantRecord

# Build a toy feature matrix from scratch to see what the baseline works with
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

# Toy: 5 mutants of length 10
toy_seqs = [
    "ACDEFGHIKL",
    "ACDEFGHIML",  # position 8: K→M
    "ACDEFRHIKL",  # position 5: G→R
    "ACDAFGHIKL",  # position 3: E→A
    "ACDEFGHIKW",  # position 9: L→W
]

records = [
    MutantRecord(seq=s, target=float(i), pll=-50.0 + i, spurs_ddg=-0.5 * i)
    for i, s in enumerate(toy_seqs)
]

X, y = _FeatureBuilder().build(records)
print(f"Feature matrix X shape: {X.shape}")
print(f"  = {len(toy_seqs[0])} positions × 20 AAs  +  1 (pll)  +  1 (spurs_ddg)")
print(f"  = {len(toy_seqs[0])}×20 + 2 = {len(toy_seqs[0])*20+2}")
print(f"\ntarget vector y: {y}")
print(f"one-hot region non-zero count per row: {(X[:, :200] > 0).sum(axis=1)}")

In [ ]:
from confit.baselines.spurs_ridge import _LowNEvaluator
from sklearn.linear_model import Ridge
import numpy as np

# Simulate the low-N evaluation loop on toy data
np.random.seed(42)
N_total = 300
D = 100  # feature dims

# Ground truth is a linear function of the first 10 features + noise
true_weights = np.random.randn(10)
X_toy = np.random.randn(N_total, D).astype(np.float32)
y_toy = X_toy[:, :10] @ true_weights + 0.5 * np.random.randn(N_total)
y_toy = y_toy.astype(np.float32)

evaluator = _LowNEvaluator(n_repeats=5, random_seed=42, alpha=1e-8)
results = evaluator.evaluate(X_toy, y_toy)

print("\nAs N increases, the model gets better:")
for r in results:
    bar = "█" * int(r.mean_spearman * 40)
    print(f"  N={r.n_train:>4d}  ρ={r.mean_spearman:.4f} ± {r.std_spearman:.4f}  {bar}")

---
## 12 · `metrics/` + `utils/` — Spearman, seeding, cleanup

**Files:** `confit/metrics/correlation.py`, `confit/utils/seeding.py`, `confit/utils/cleanup.py`

In [ ]:
from confit.metrics.correlation import spearman

# spearman() wraps scipy.stats.spearmanr and returns just the correlation value
a = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("Perfect positive  :", spearman(a, a))           # 1.0
print("Perfect negative  :", spearman(a, -a))          # -1.0
print("Monotone but noisy:", spearman(a, a + np.random.randn(5) * 0.5))

In [ ]:
from confit.utils.seeding import seed_everything

# seed_everything sets:
#   - Python random
#   - NumPy random
#   - torch.manual_seed (controls weight init)
#   - torch.cuda.manual_seed_all
#   - cudnn deterministic=True, benchmark=False
#
# sample_seed and model_seed are intentionally separate:
#   sample_seed controls which k training examples are selected
#   model_seed  controls weight initialisation AND which ESM-1v seed (1-5) to load

seed_everything(sample_seed=0, model_seed=1)

x1 = torch.randn(3)
seed_everything(sample_seed=0, model_seed=1)
x2 = torch.randn(3)
print("Reproducible tensors after re-seeding:", torch.allclose(x1, x2))  # True

In [ ]:
from confit.utils.cleanup import ArtifactCleaner

# ArtifactCleaner removes old predicted_* and checkpoint_* directories.
# dry_run=True prints what would be deleted without touching anything.
#
# clean_predicted() — removes per-seed pred.csv folders under predicted_*/
# clean_checkpoint() — removes checkpoint folders under checkpoint_*/
#
# To run for real:
# cleaner = ArtifactCleaner(base_dir=Path("."), dry_run=False)
# cleaner.clean_predicted()
# cleaner.clean_checkpoint()

cleaner = ArtifactCleaner(base_dir=Path("."), dry_run=True)
print("dry_run:", cleaner.dry_run)
print("base_dir:", cleaner.base_dir)

# Preview what clean_predicted would touch:
print("\nDRY RUN — predicted_* folders found:")
for d in sorted(Path(".").glob("predicted_*")):
    print(" ", d)

---
## Summary: the full pipeline in one picture

```
config/training_config.yaml
        │
        ▼
ConfigLoader.load()  →  TrainingConfig (validated, frozen)
        │
        ▼
ModelRegistry  →  hub_id  →  ESMModelFactory.build()
                                    │
                      ┌─────────────┼─────────────┐
                  backbone       reg_model     tokenizer
                  (trainable)    (frozen)
                      │
              ConFitTrainer.build_peft_model()  ← LoRA applied here
                      │
                      ▼
            MutationDataset  →  DataLoader
                      │
                      ▼
        ┌─────────────────────────────┐
        │  Per training step:         │
        │  MaskedMarginalScorer.score()│
        │    → scores, logits         │
        │  BradleyTerryLoss(scores)   │
        │  KLRegularizationLoss(logits│
        │  total = BT + λ*KL          │
        │  AModule learns 'a'         │
        └─────────────────────────────┘
                      │
              checkpoint_*/
                      │
                      ▼
            ConFitEvaluator  →  pred.csv
                      │
                      ▼
         InferenceAggregator  →  ensemble average  →  Spearman ρ
```